# model_experiment_Prophet

Prophet (Taylor & Letham, Facebook 2017) — an additive (GAM-style) model per series:

`y(t) = trend(t) + seasonality(t) + holidays(t) + noise`

- **trend** — piecewise-linear with automatically detected changepoints;
- **seasonality** — Fourier series (yearly here; weekly/daily off for weekly data);
- **holidays** — explicit regressors for our big-4 holiday weeks, Prophet's distinctive advantage over ARIMA-family models.

Unlike SARIMA (~3.5s/series at s=52), Prophet's MAP fit takes ~0.3s/series, so we fit it on **every series with ≥60 weeks of history** (~2900 series, ~17 min) instead of a top-50 hybrid — still per-series (no cross-series learning), which remains the family's structural limitation.

MLflow experiment: **Prophet_Training**. CPU; full Run All ≈ 40 min.

In [ ]:
# Kaggle bootstrap — does nothing when running locally.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow prophet")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [ ]:
import sys, pathlib, time, logging
sys.path.insert(0, str(pathlib.Path().resolve()))
logging.getLogger("cmdstanpy").disabled = True   # else 2 log lines per fit x 3000 fits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from prophet import Prophet

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import split_fold
from src.mlflow_utils import setup_mlflow
from src.postprocess import naive_lag52, apply_christmas_shift, blend_holiday_naive
from src.preprocessing import (SUPER_BOWL, LABOR_DAY, THANKSGIVING, CHRISTMAS,
                               BLEND_HOLIDAY_WEEKS)

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
setup_mlflow("Prophet_Training")

HOLIDAYS = pd.concat([
    pd.DataFrame({"holiday": "super_bowl", "ds": SUPER_BOWL}),
    pd.DataFrame({"holiday": "labor_day", "ds": LABOR_DAY}),
    pd.DataFrame({"holiday": "thanksgiving", "ds": THANKSGIVING}),
    pd.DataFrame({"holiday": "christmas", "ds": CHRISTMAS}),
])


def prophet_forecast_all(history, target_dates, min_obs=60):
    """Fit Prophet per series (>= min_obs observations), return long pred df."""
    future = pd.DataFrame({"ds": pd.DatetimeIndex(target_dates)})
    counts = history.groupby(["Store", "Dept"]).size()
    eligible = counts[counts >= min_obs].index
    rows = []
    t0 = time.time()
    for k, (st, dp) in enumerate(eligible):
        y = history[(history.Store == st) & (history.Dept == dp)][
            ["Date", "Weekly_Sales"]].rename(columns={"Date": "ds", "Weekly_Sales": "y"})
        try:
            m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                        daily_seasonality=False, holidays=HOLIDAYS)
            m.fit(y)
            fc = m.predict(future)
            rows.append(pd.DataFrame({"Store": st, "Dept": dp,
                                      "Date": fc["ds"], "pred": fc["yhat"]}))
        except Exception:
            pass
        if (k + 1) % 500 == 0:
            print(f"  {k+1}/{len(eligible)} series ({time.time()-t0:.0f}s)")
    print(f"fitted {len(rows)}/{len(eligible)} series in {(time.time()-t0)/60:.1f} min")
    return pd.concat(rows)


def assemble(pred_long, history, target):
    m = target.merge(pred_long, on=["Store", "Dept", "Date"], how="left")
    coverage = m["pred"].notna().mean()
    m["pred"] = m["pred"].fillna(pd.Series(naive_lag52(history, target), index=m.index))
    dep_med = history.groupby("Dept")["Weekly_Sales"].median()
    m["pred"] = m["pred"].fillna(m["Dept"].map(dep_med)).fillna(0)
    return m, coverage

## Run 1 — Anatomy of one fit: components plot (Store 20 / Dept 92)

In [ ]:
tr, va = split_fold(train, 1)
s = tr[(tr.Store == 20) & (tr.Dept == 92)][["Date", "Weekly_Sales"]].rename(
    columns={"Date": "ds", "Weekly_Sales": "y"})
m_demo = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                 daily_seasonality=False, holidays=HOLIDAYS)
m_demo.fit(s)
fc_demo = m_demo.predict(pd.DataFrame({"ds": sorted(va["Date"].unique())}))
m_demo.plot_components(fc_demo)
plt.show()

with mlflow.start_run(run_name="Prophet_Demo"):
    mlflow.log_params({"series": "store20_dept92",
                       "components": "piecewise trend + yearly Fourier + 4 holiday regressors"})

## Run 2 — Fold-1 evaluation: Prophet on ALL eligible series (~17 min)

In [ ]:
pred_va = prophet_forecast_all(tr, sorted(va["Date"].unique()))
m1, coverage = assemble(pred_va, tr, va)
rep = wmae_report(m1["Weekly_Sales"], m1["pred"], m1["IsHoliday"])
with mlflow.start_run(run_name="Prophet_CV_Fold1"):
    mlflow.log_params({"min_obs": 60, "holidays": "big-4 regressors",
                       "fallback": "seasonal naive + dept median", "fold": 1})
    mlflow.log_metrics({**rep, "coverage": coverage})
print(f"coverage {coverage:.3f}, {rep}")

## Run 3 — Holiday blend check

In [ ]:
blend_scores = {}
for w in (0.0, 0.25, 0.5):
    blended = blend_holiday_naive(m1[["Store", "Dept", "Date", "pred"]], tr,
                                  weight=w, holiday_dates=BLEND_HOLIDAY_WEEKS)
    r = wmae_report(m1["Weekly_Sales"], blended["pred"], m1["IsHoliday"])
    blend_scores[w] = r["wmae"]
    with mlflow.start_run(run_name=f"Prophet_Blend_noXmas_w{w}"):
        mlflow.log_params({"blend_weight": w, "fold": 1})
        mlflow.log_metrics(r)
    print(f"w={w}: {r}")
BLEND_W = min(blend_scores, key=blend_scores.get)
print("best blend weight:", BLEND_W)

## Run 4 — Final: full train -> test submission (~20 min)

In [ ]:
pred_test = prophet_forecast_all(train, sorted(test["Date"].unique()))
sub, coverage = assemble(pred_test, train, test[["Store", "Dept", "Date"]])
sub = sub[["Store", "Dept", "Date", "pred"]]
sub = apply_christmas_shift(sub, pred_col="pred")
if BLEND_W > 0:
    sub = blend_holiday_naive(sub, train, weight=BLEND_W, holiday_dates=BLEND_HOLIDAY_WEEKS)

with mlflow.start_run(run_name="Prophet_Final"):
    mlflow.log_params({"min_obs": 60, "blend_weight": BLEND_W,
                       "post": "christmas_shift + noXmas_blend"})
    mlflow.log_metrics({"fold1_wmae": rep["wmae"], "test_coverage": coverage})

make_submission(sub, "pred", "submission_prophet.csv")
print(f"wrote submission_prophet.csv (coverage {coverage:.3f}, blend w={BLEND_W})")